**ATTENTION AND TRANSFORMERS**

**TRANSFORMER ARCHITECTURE FROM NUMPY**

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
x = np.random.randn(5,4)
x

array([[ 0.58947594, -1.91139021,  0.5018783 , -0.13070056],
       [-0.10631518, -1.46311076,  1.27330403,  0.11992461],
       [ 0.75767802,  0.95895246, -0.34042803,  1.20360025],
       [ 0.07083349,  0.53908092, -0.59291082,  0.60386828],
       [-0.68868547,  0.08922388, -1.22822076,  0.93377778]])

In [3]:
wq = np.random.rand(4,4)
wk = np.random.rand(4,4)
wv = np.random.rand(4,4)

In [4]:
Q = np.dot(x,wq)
K = np.dot(x,wk)
V = np.dot(x,wv)

In [5]:
raw_score = (Q @ K.T)/np.sqrt(4)

In [6]:
raw_score.shape

(5, 5)

In [7]:
softmax = np.exp(raw_score) / np.sum(np.exp(raw_score),axis=1,keepdims=True)

In [8]:
output = np.dot(softmax,V)
output.shape

(5, 4)

In [9]:
print(np.sum(softmax, axis=1))

[1. 1. 1. 1. 1.]


**ATTENTION AND TRANSFORMER APPLIED TO JPMC AND S&P500 DATA**

**PREPARING THE DATA**

In [10]:
jpmc = yf.download(['JPM'],start = '2010-01-1',end='2025-01-01')
sp500 = yf.download(['^GSPC'],start = '2010-01-1',end='2025-01-01')

/tmp/ipykernel_3746/4128732890.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  jpmc = yf.download(['JPM'],start = '2010-01-1',end='2025-01-01')
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_3746/4128732890.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download(['^GSPC'],start = '2010-01-1',end='2025-01-01')
[*********************100%***********************]  1 of 1 completed


**EXPLORATORY DATA ANALYSIS**

In [11]:
jpmc.isnull().sum().sum()

np.int64(0)

In [12]:
sp500.isnull().sum().sum()

np.int64(0)

In [13]:
jpmc.duplicated().sum()

np.int64(0)

In [14]:
sp500.duplicated().sum()

np.int64(0)

In [15]:
jpmc.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3774 entries, 2010-01-04 to 2024-12-31
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (Close, JPM)   3774 non-null   float64
 1   (High, JPM)    3774 non-null   float64
 2   (Low, JPM)     3774 non-null   float64
 3   (Open, JPM)    3774 non-null   float64
 4   (Volume, JPM)  3774 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 176.9 KB


In [16]:
sp500.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3774 entries, 2010-01-04 to 2024-12-31
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   (Close, ^GSPC)   3774 non-null   float64
 1   (High, ^GSPC)    3774 non-null   float64
 2   (Low, ^GSPC)     3774 non-null   float64
 3   (Open, ^GSPC)    3774 non-null   float64
 4   (Volume, ^GSPC)  3774 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 176.9 KB


**FEATURE ENGINEERING**

In [17]:
#Daily returns
jpmc['returns'] = jpmc['Close']['JPM'].pct_change(fill_method=None)

In [18]:
#volume ratio
volume = jpmc['Volume']['JPM']
jpmc['volume ratio'] = (volume/ volume.rolling(window=20).mean()).shift(1)

In [19]:
#20-day rolling return
jpmc['20-day rolling return'] = jpmc['returns'].rolling(window=20).mean().shift(1)

In [20]:
delta = jpmc['Close']['JPM'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
RSI = 100 - (100 / (1 + gain/loss))
jpmc['RSI'] = RSI.shift(1)

In [21]:
sp500['SP500 returns'] = sp500['Close']['^GSPC'].pct_change(fill_method=None)

In [22]:
sp500_return_dataframe = pd.DataFrame({
    'Date': sp500.index,
    'SP500 returns': sp500['SP500 returns']
})
sp500_return_dataframe.set_index('Date', inplace=True)

In [23]:
jpmc_clean = pd.DataFrame({
    "returns": jpmc['returns'],
    "Volume Ratio": jpmc['volume ratio'],
    "Rolling Returns": jpmc['20-day rolling return'],
    "RSI": jpmc['RSI']
})

In [24]:
jpmc_clean = jpmc_clean.join(sp500_return_dataframe,how='inner')

In [25]:
jpmc_clean['target'] = (jpmc_clean['returns'] > 0).astype(int).shift(-1)

In [26]:
jpmc_clean.dropna(inplace=True)
jpmc_clean

,returns,Volume Ratio,Rolling Returns,RSI,SP500 returns,target
Date,,,,,,
2010-02-03,-0.006412,0.853654,-0.002507,36.612158,-0.005474,0.0
2010-02-04,-0.048151,0.696517,-0.003796,31.106907,-0.031141,0.0
2010-02-05,-0.001304,1.037733,-0.006478,23.539295,0.002897,0.0
2010-02-08,-0.015665,1.327887,-0.007534,25.589869,-0.008863,1.0
2010-02-09,0.018302,1.007097,-0.008194,25.133765,0.013040,1.0
...,...,...,...,...,...,...
2024-12-23,0.003325,3.523394,-0.001417,36.638831,0.007287,1.0
2024-12-24,0.016444,0.934839,-0.002024,39.867638,0.011043,1.0
2024-12-26,0.003425,0.419782,-0.001552,48.407826,-0.000406,0.0


In [27]:
jpmc_clean.shape

(3752, 6)

In [28]:
jpmc_clean.columns

Index(['returns', 'Volume Ratio', 'Rolling Returns', 'RSI', 'SP500 returns',
       'target'],
      dtype='object')

**SEQUENCES AND SPLITTING DATA**

In [29]:
features = jpmc_clean.drop('target',axis=1)
target = jpmc_clean['target']

In [30]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

In [31]:
target = target.to_numpy()

In [32]:
lookback = 20
x,y = [],[]

for i in range (len(features_scaled) - lookback):
  x.append(features_scaled[i:i+lookback])
  y.append(target[i+lookback])

In [33]:
x = np.array(x)
y = np.array(y)

In [34]:
x.shape

(3732, 20, 5)

In [35]:
y.shape

(3732,)

In [36]:
x_train = x[:int(0.8*len(x))]
y_train = y[:int(0.8*len(y))]
x_test = x[int(0.8*len(x)):]
y_test = y[int(0.8*len(y)):]

In [37]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(2985, 20, 5)
(2985,)
(747, 20, 5)
(747,)


In [39]:
import torch
import torch.nn as nn